# 10 — Relatório de Utilização por Especialidade

Relatório gerencial sobre o uso do assistente clínico, alinhado ao requisito **"Relatórios de utilização por especialidade médica"** da Fase 3.

## Fontes de dados

- `hospital.db` (SQLite mock) — tabelas `pacientes`, `exames`, `registros_violencia`, `log_acesso`
- Nenhum dado externo: relatório auto-contido, lê do estado atual da base mock

## Saídas

1. Distribuição da base de pacientes (faixa etária, convênio)
2. Cobertura de rastreamentos preventivos por categoria
3. Auditoria de acessos aos registros sensíveis (LGPD)
4. Padrões de uso por especialidade médica simulada
5. Indicadores de qualidade do atendimento

## Como rodar

- **Colab:** monta Drive, usa `hospital.db` em `/MyDrive/AssistenteHospitalar/files/`
- **Local:** define `HOSPITAL_DB_PATH=/caminho/local/hospital.db` antes de executar

In [ ]:
!pip install -q pandas matplotlib

In [ ]:
import os, sys, sqlite3
from pathlib import Path
import pandas as pd

# Auto-detecta ambiente: Colab monta Drive, local usa env var
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DB_PATH = '/content/drive/MyDrive/AssistenteHospitalar/files/hospital.db'
else:
    DB_PATH = os.environ.get(
        'HOSPITAL_DB_PATH',
        '/content/drive/MyDrive/AssistenteHospitalar/files/hospital.db',
    )

if not Path(DB_PATH).exists():
    raise FileNotFoundError(
        f'hospital.db não encontrado em {DB_PATH}. '
        'Rode 05_gerar_dados_mock.ipynb antes.'
    )

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 140)
print(f'Conectado a: {DB_PATH}')

## 1. Distribuição da base de pacientes

In [ ]:
# Total + distribuição por faixa etária e convênio
pacientes = pd.read_sql_query(
    """
    SELECT paciente_id, nome, data_nascimento, convenio,
           CAST((julianday('2026-05-25') - julianday(data_nascimento)) / 365.0 AS INTEGER) AS idade
    FROM pacientes
    """,
    conn,
)

def faixa_etaria(idade):
    if idade < 25: return '18-24'
    if idade < 40: return '25-39'
    if idade < 50: return '40-49'
    if idade < 65: return '50-64'
    return '65+'

pacientes['faixa'] = pacientes['idade'].apply(faixa_etaria)

print(f'Total de pacientes: {len(pacientes)}\n')

print('Distribuição por faixa etária:')
print(pacientes['faixa'].value_counts().sort_index().to_string())

print('\nDistribuição por convênio:')
print(pacientes['convenio'].value_counts().to_string())

## 2. Cobertura de rastreamento preventivo

Indicador de qualidade do serviço — quanto da população elegível está em dia com mamografia (50-69a bienal) e papanicolau (25-64a trienal), seguindo regras do MS.

In [ ]:
from datetime import date
TODAY = date(2026, 5, 25)

def cobertura(tipo, idade_min, idade_max, anos_intervalo):
    elegiveis = pacientes[(pacientes['idade'] >= idade_min) & (pacientes['idade'] <= idade_max)]
    total = len(elegiveis)
    em_dia = 0
    for pid in elegiveis['paciente_id']:
        r = conn.execute(
            'SELECT MAX(data_realizacao) AS dt FROM exames WHERE paciente_id=? AND tipo=?',
            (int(pid), tipo),
        ).fetchone()
        if r['dt']:
            ultima = date.fromisoformat(r['dt'])
            if (TODAY - ultima).days <= anos_intervalo * 365 + 180:
                em_dia += 1
    pct = (em_dia / total * 100) if total else 0
    print(f"  {tipo:<15} elegíveis={total:>3}  em dia={em_dia:>3}  cobertura={pct:5.1f}%")

print('Cobertura de rastreamento populacional:\n')
cobertura('papanicolau', 25, 64, 3)
cobertura('mamografia',  50, 69, 2)

## 3. Auditoria LGPD — acesso a dados sensíveis

Trilha de auditoria de toda interação com `registros_violencia` (consulta + escrita).  
Cumpre o requisito **"Auditoria de acesso a dados sensíveis"** + **"Logs específicos para casos de violência doméstica"**.

In [ ]:
logs = pd.read_sql_query(
    'SELECT timestamp, usuario, tabela, paciente_id, motivo FROM log_acesso ORDER BY timestamp DESC',
    conn,
)
print(f'Total de acessos auditados: {len(logs)}\n')

if len(logs):
    print('Acessos por usuário:')
    print(logs['usuario'].value_counts().to_string())

    print('\nAcessos por tabela:')
    print(logs['tabela'].value_counts().to_string())

    print('\nÚltimos 10 acessos:')
    display(logs.head(10))
else:
    print('(Nenhum acesso registrado ainda — interaja com a UI ou workflows para gerar trilha.)')

## 4. Padrões de uso por especialidade médica

Distribuição de exames preventivos realizados, agrupados por tipo (proxy de especialidade): citologia → ginecologia ambulatorial; mamografia → mastologia; USG pélvica → gineco-obstetrícia, etc.

In [ ]:
ESPECIALIDADE_POR_TIPO = {
    'papanicolau':   'Ginecologia ambulatorial',
    'mamografia':    'Mastologia / radiologia',
    'usg_pelvica':   'Ginecologia / Obstetrícia',
    'usg_mamaria':   'Mastologia',
    'colposcopia':   'Ginecologia (alta complexidade)',
}

exames = pd.read_sql_query(
    'SELECT tipo, data_realizacao, resultado FROM exames',
    conn,
)
exames['especialidade'] = exames['tipo'].map(ESPECIALIDADE_POR_TIPO).fillna('Outras')

print('Exames realizados por especialidade:')
print(exames['especialidade'].value_counts().to_string())

print('\nDistribuição por tipo:')
print(exames['tipo'].value_counts().to_string())

## 5. Indicadores de violência doméstica

Métricas agregadas (sem expor dados nominativos) do volume e tipologia dos registros.  
Útil para acompanhamento epidemiológico e dimensionamento de equipe especializada.

In [ ]:
registros = pd.read_sql_query(
    'SELECT tipo, data_atendimento, notificado_sinan, encaminhamentos FROM registros_violencia',
    conn,
)

print(f'Total de registros de violência: {len(registros)}\n')

if len(registros):
    print('Por tipo:')
    print(registros['tipo'].value_counts().to_string())

    pct_sinan = (registros['notificado_sinan'].sum() / len(registros) * 100)
    print(f'\nNotificação SINAN: {int(registros["notificado_sinan"].sum())} de {len(registros)} '
          f'({pct_sinan:.0f}%)')

    print('\nEncaminhamentos mais frequentes (top 5):')
    print(registros['encaminhamentos'].value_counts().head(5).to_string())

## 6. Indicadores de alerta — pacientes em atraso de rastreamento

Lista (limitada a 10) das pacientes com exame preventivo mais atrasado.  
Suporte a campanhas ativas de busca / agendamento prioritário.

In [ ]:
# Pacientes 50-69a com mamografia mais atrasada (ou sem nenhuma)
mamografia_atraso = pd.read_sql_query(
    """
    SELECT p.paciente_id,
           p.nome,
           CAST((julianday('2026-05-25') - julianday(p.data_nascimento)) / 365.0 AS INTEGER) AS idade,
           MAX(e.data_realizacao) AS ultima_mamo,
           CAST((julianday('2026-05-25') - julianday(MAX(e.data_realizacao))) / 365.0 AS INTEGER) AS anos_desde_ultimo
    FROM pacientes p
    LEFT JOIN exames e ON e.paciente_id = p.paciente_id AND e.tipo = 'mamografia'
    WHERE CAST((julianday('2026-05-25') - julianday(p.data_nascimento)) / 365.0 AS INTEGER) BETWEEN 50 AND 69
    GROUP BY p.paciente_id
    ORDER BY (ultima_mamo IS NULL) DESC, ultima_mamo ASC
    LIMIT 10
    """,
    conn,
)
print('Top 10 pacientes 50-69a com mamografia atrasada / nunca realizada:')
display(mamografia_atraso)

## 7. Resumo executivo (síntese)

In [ ]:
def conta(t):
    return conn.execute(f'SELECT COUNT(*) AS c FROM {t}').fetchone()['c']

print('=' * 60)
print('RESUMO EXECUTIVO — Assistente Clínico Hospitalar')
print('=' * 60)
print(f'Pacientes cadastradas:          {conta("pacientes"):>6}')
print(f'Prontuários ginecológicos:      {conta("prontuario_gineco"):>6}')
print(f'Exames preventivos registrados: {conta("exames"):>6}')
print(f'Ciclos menstruais registrados:  {conta("ciclos_menstruais"):>6}')
print(f'Registros de violência:         {conta("registros_violencia"):>6}')
print(f'Medicamentos cadastrados:       {conta("medicamentos"):>6}')
print(f'Acessos auditados (LGPD):       {conta("log_acesso"):>6}')
print('=' * 60)

In [ ]:
conn.close()
print('Relatório concluído.')